# ArchiCheck — Extracción Geométrica (Plantilla)

**Usar para cada proyecto nuevo. Solo editar la Celda 3.**

1. Celda 1 — instala dependencias base
2. Celda 2 — sube el PDF del proyecto
3. **Celda 3 — configura páginas y escalas** ← única celda que editas
4. Celda 4 — OpenCV: recintos, áreas, anchos
5. **Celda 4b — carga Grounding DINO + SAM 2** ← ejecutar UNA VEZ por sesión (requiere GPU T4)
6. Celda 4c — detección semántica de elementos (puertas, ventanas, escaleras, rampas)
7. Celda 5 — visualización con detecciones superpuestas
8. Celda 6 — informe consola + guardar JSON
9. Celda 7 — descargar resultados

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 1 — Instalar dependencias (~60 segundos)
# ══════════════════════════════════════════════════════════
print('Instalando librerías...')
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pymupdf', 'opencv-python-headless', 'matplotlib',
                'Pillow', 'numpy', 'requests'], check=True)
print('✓ Listo')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 2 — Subir PDF, validar que sea vectorizado y convertir a imágenes
# ══════════════════════════════════════════════════════════
import fitz
import numpy as np
import cv2
from google.colab import files

print('Selecciona el PDF del plano cuando aparezca el botón:')
uploaded  = files.upload()
pdf_name  = list(uploaded.keys())[0]
pdf_bytes = uploaded[pdf_name]

ZOOM = 3
DPI  = 72 * ZOOM

doc = fitz.open(stream=pdf_bytes, filetype='pdf')
print(f'\nPDF: "{pdf_name}" — {len(doc)} página(s)')

# ── VALIDACIÓN: el PDF debe ser vectorizado, no un escaneo ─────
# FIX 2026-07-23: se detecto que el PDF de prueba (Plaza Pedro de Valdivia) es
# vectorizado — 248 items de texto real con coordenadas exactas y 3722 trazos
# vectoriales en una sola pagina. Eso abre la puerta a extraer cotas y lineas
# de muro/puerta/ventana como datos exactos en vez de adivinar desde pixeles.
# Pero esto SOLO funciona si el PDF es vectorizado. Un escaneo o foto del plano
# impreso no tiene texto ni trazos extraibles — hay que rechazarlo ahora mismo,
# antes de gastar tiempo de GPU/API en un archivo que no va a rendir bien.
print('\nValidando que el PDF sea vectorizado (no un escaneo)...')
total_text_len  = 0
total_drawings  = 0
for _pg in doc:
    total_text_len += len(_pg.get_text('text').strip())
    total_drawings += len(_pg.get_drawings())

print(f'  Texto extraído del PDF : {total_text_len} caracteres')
print(f'  Trazos vectoriales     : {total_drawings}')

MIN_TEXT_CHARS = 50
MIN_DRAWINGS   = 20
if total_text_len < MIN_TEXT_CHARS or total_drawings < MIN_DRAWINGS:
    raise ValueError(
        "\n⛔  Este PDF parece ser un ESCANEO o imagen rasterizada, no un PDF vectorizado.\n"
        f"    Texto extraído: {total_text_len} caracteres (mínimo {MIN_TEXT_CHARS})\n"
        f"    Trazos vectoriales: {total_drawings} (mínimo {MIN_DRAWINGS})\n\n"
        "    Por ahora ArchiCheck solo procesa PDF vectorizados, exportados directo\n"
        "    desde el software de diseño (AutoCAD, Revit, ArchiCAD, etc. → 'Exportar a PDF'),\n"
        "    no un escaneo, foto o PDF impreso-y-vuelto-a-escanear del plano.\n"
        "    Pedile al arquitecto el PDF vectorizado original y volvé a subirlo.\n"
    )
print('  ✓ PDF vectorizado confirmado — continuando.\n')

paginas = []
for i, page in enumerate(doc):
    mat     = fitz.Matrix(ZOOM, ZOOM)
    pix     = page.get_pixmap(matrix=mat, alpha=False)
    buf     = np.frombuffer(pix.tobytes('png'), np.uint8)
    img     = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    paginas.append(img_rgb)
    print(f'  Página {i+1}: {img_rgb.shape[1]}x{img_rgb.shape[0]} px')

print(f'\n✓ {len(paginas)} página(s) cargadas')
print('Revisa los números de página arriba y configura la Celda 3.')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 3 — CONFIGURACIÓN  ← EDITA AQUÍ ANTES DE EJECUTAR
# ══════════════════════════════════════════════════════════
import re, matplotlib.pyplot as plt, matplotlib.patches as patches_plt
from datetime import datetime

# ── NOMBRE DEL PROYECTO ───────────────────────────────────
# Nombre corto del cliente o proyecto. Sin espacios.
# Usa solo letras, números y guiones bajos.
# Ejemplos: 'pdv', 'beaucheff', 'casa_garcia', 'edificio_centro'

NOMBRE_PROYECTO = ''   # ← COMPLETAR

# ── PÁGINAS Y ESCALAS ─────────────────────────────────────
#
# Formato básico:   (numero_pagina, 'escala')
# Con recorte:      (numero_pagina, 'escala', (x1, y1, x2, y2))
#
# x1, y1, x2, y2 son fracciones 0.0–1.0 de la imagen completa:
#   (0.0, 0.0, 1.0, 1.0)  → toda la página  (igual a no poner recorte)
#   (0.0, 0.0, 0.5, 1.0)  → mitad izquierda
#   (0.5, 0.0, 1.0, 0.5)  → cuadrante superior derecho
#
# Escalas comunes: '1:25'  '1:50'  '1:75'  '1:100'  '1:200'  '1:500'
#
# IMPORTANTE: incluye solo secciones con recintos cerrados (plantas).
# Si una página tiene planta + elevación, recorta solo la zona de planta.
# Omite páginas de carátula, especificaciones, elevaciones o detalles sin recintos.
#
# Ejemplos:
#   (2, '1:50')                           → página 2 completa, escala 1:50
#   (3, '1:50', (0.0, 0.0, 0.55, 1.0))   → mitad izquierda de página 3 (planta)
#   (1, '1:100', (0.0, 0.0, 1.0, 0.5))   → mitad superior de página 1

PAGINAS_Y_ESCALAS = [
    # (N°, 'escala'),
    # (N°, 'escala', (x1, y1, x2, y2)),
]

# ─────────────────────────────────────────────────────────
# No edites nada bajo esta línea

# Validar nombre de proyecto
if not NOMBRE_PROYECTO.strip():
    raise ValueError(
        "\n⛔  NOMBRE_PROYECTO está vacío.\n"
        "    Completa la variable antes de continuar. Ejemplo: NOMBRE_PROYECTO = 'pdv'\n"
    )

_slug = re.sub(r'[^a-z0-9]+', '_', NOMBRE_PROYECTO.strip().lower()).strip('_') or 'proyecto'
MESES_ES = ['ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']
_now      = datetime.now()
TIMESTAMP = f'{_now.day:02d}{MESES_ES[_now.month-1]}_{_now.strftime("%H%M")}'
BASENAME  = f'archicheck_geometrico_{_slug}_{TIMESTAMP}'

print(f'  Proyecto  : {NOMBRE_PROYECTO}')
print(f'  Timestamp : {TIMESTAMP}')
print(f'  Nombre base de archivos: {BASENAME}')

if not PAGINAS_Y_ESCALAS:
    print('\n⚠ PAGINAS_Y_ESCALAS está vacía — agrega al menos una página antes de continuar.')
else:
    entries = [(e[0], e[1], e[2] if len(e) > 2 else None) for e in PAGINAS_Y_ESCALAS]
    n_pags  = len(entries)
    fig, axes = plt.subplots(1, n_pags, figsize=(14 * n_pags, 9))
    if n_pags == 1:
        axes = [axes]
    for ax, (pag, esc, crop) in zip(axes, entries):
        if pag < 1 or pag > len(paginas):
            ax.set_title(f'Página {pag} — NO EXISTE en el PDF', color='red')
            ax.axis('off')
            continue
        plano_full = paginas[pag - 1]
        h_f, w_f   = plano_full.shape[:2]
        scale_ratio = int(esc.split(':')[1])
        MPX   = 0.0254 * scale_ratio / DPI
        ax.imshow(plano_full)
        if crop:
            x1f, y1f, x2f, y2f = crop
            rect = patches_plt.Rectangle(
                (x1f * w_f, y1f * h_f),
                (x2f - x1f) * w_f, (y2f - y1f) * h_f,
                linewidth=3, edgecolor='#E74C3C', facecolor='#E74C3C22'
            )
            ax.add_patch(rect)
            crop_txt = f'  recorte ({x1f:.0%},{y1f:.0%})→({x2f:.0%},{y2f:.0%})'
            h_crop = int((y2f - y1f) * h_f)
            w_crop = int((x2f - x1f) * w_f)
        else:
            crop_txt = '  sin recorte'
            h_crop, w_crop = h_f, w_f
        ax.set_title(
            f'Página {pag} — escala {esc}\n'
            f'{w_crop}x{h_crop} px analizados  |  {MPX:.5f} m/px{crop_txt}',
            fontsize=10
        )
        ax.axis('off')
    plt.suptitle(
        f'{NOMBRE_PROYECTO} — vista previa páginas a analizar (zona roja = recorte activo)',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
    print(f'✓ {n_pags} entrada(s) configurada(s). Si el preview es correcto, ejecuta la Celda 4.')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 4 — PROCESAR TODAS LAS PÁGINAS
#   Por cada página: Claude Vision + extracción vectorial + OpenCV + cruce
# ══════════════════════════════════════════════════════════
import base64, json, requests, re, math
import cv2, numpy as np
from datetime import datetime

WORKER_URL = 'https://archicheck-worker.nestragues.workers.dev'

OGUC_REGLAS = {
    'dormitorio': (8.0,  None, 'Art. 4.1.7 OGUC'),
    'sala'      : (10.0, None, 'Art. 4.1.7 OGUC'),
    'living'    : (10.0, None, 'Art. 4.1.7 OGUC'),
    'comedor'   : (8.0,  None, 'Art. 4.1.7 OGUC'),
    'cocina'    : (3.0,  None, 'Art. 4.1.7 OGUC'),
    'bano'      : (1.5,  None, 'Art. 4.1.7 OGUC'),
    'pasillo'   : (None, 1.20, 'Art. 4.2.2 OGUC — ancho min 1.20 m'),
    'escalera'  : (None, 1.20, 'Art. 4.2.4 OGUC — ancho min 1.20 m'),
    'rampa'     : (None, 1.20, 'Art. 4.1.7 OGUC + DS 50/2015'),
}

def mejorar_contraste_nitidez(img_rgb):
    """
    CAMBIO 2026-07-23: se descarto la hipotesis de que el 0% de deteccion de
    ventanas fuera un problema de prompt (se probo en vivo, ver roadmap P1) —
    pero el contraste/nitidez de la imagen sigue siendo sospechoso: los
    simbolos de ventana son lineas finas dentro de un vano de muro, y el
    relleno de color de los recintos reduce el contraste justo ahi.
    Esta funcion NO reemplaza la imagen original — genera una segunda version
    con CLAHE (contraste adaptativo) + nitidez, que se manda como imagen
    adicional a Claude, no en vez de la original.
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    contraste = clahe.apply(gray)
    kernel_nitidez = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    nitido = cv2.filter2D(contraste, -1, kernel_nitidez)
    return cv2.cvtColor(nitido, cv2.COLOR_GRAY2RGB)

def _es_trazo_discontinuo(path):
    """
    NUEVO 2026-07-23. Un trazo es 'discontinuo' (linea de referencia: deslinde,
    linea de edificacion, eje) si el PDF lo define con un patron de guiones
    real (path['dashes'], formato PDF tipo '[3 2] 0'). Formato solido es '[] 0'
    o vacio. Esto es una señal exacta del vector, no una heuristica de pixeles
    sobre grosor o densidad.
    """
    dashes = (path.get('dashes') or '').strip()
    if not dashes:
        return False
    contenido = dashes.split(']')[0].replace('[', '').strip()
    return len(contenido) > 0

def extraer_datos_vectoriales(pdf_page, zoom, mpx, crop_px=None, max_largo_trazo_m=3.0):
    """
    Extrae texto y trazos vectoriales directamente del PDF (objeto fitz.Page),
    en vez de adivinarlos desde pixeles. Convierte las coordenadas al mismo
    espacio de pixeles que usa el resto del pipeline (aplicando ZOOM), y
    recorta a crop_px = (x1,y1,x2,y2) en pixeles si se especifica.

    Filtra trazos largos (muros/limites de recinto — ya los cubre OpenCV) y
    se queda con los cortos (candidatos a simbolo: arco de puerta, lineas de
    ventana, etc.), usando max_largo_trazo_m como umbral en metros reales.

    NUEVO 2026-07-23: ademas de 'trazos' (cortos, candidatos a simbolo),
    devuelve 'lineas_discontinuas' — trazos con patron de guiones real,
    SIN filtro de largo (nos interesan aunque sean muy largos, como un
    deslinde que cruza toda la pagina), para poder borrarlos del raster
    antes de detectar recintos y que no los corten en dos.

    Retorna:
      'cotas_texto'         : [{'texto','x','y','w','h'}, ...] — reemplaza el OCR
      'trazos'               : [{'tipo':'l'|'c'|'re'|'qu','puntos':[(x,y),...],'ancho_linea'}, ...]
      'lineas_discontinuas'  : [{'puntos':[(x,y),...],'ancho_linea'}, ...]
      'n_texto', 'n_trazos', 'n_trazos_descartados_largos', 'n_lineas_discontinuas'
    """
    def to_px(pt):
        return (pt.x * zoom, pt.y * zoom)

    def dentro_crop(x, y):
        if crop_px is None:
            return True
        cx1, cy1, cx2, cy2 = crop_px
        return cx1 <= x <= cx2 and cy1 <= y <= cy2

    def ajustar(x, y):
        if crop_px is None:
            return (x, y)
        cx1, cy1, _, _ = crop_px
        return (x - cx1, y - cy1)

    # ── Texto: cotas y nombres de recintos, con posicion exacta ──
    # Reemplaza la necesidad de OCR (PaddleOCR) para PDF vectorizados.
    cotas_texto = []
    texto_dict = pdf_page.get_text('dict')
    for block in texto_dict.get('blocks', []):
        for line in block.get('lines', []):
            for span in line.get('spans', []):
                texto = span['text'].strip()
                if not texto:
                    continue
                bbox = span['bbox']  # (x0,y0,x1,y1) en puntos PDF
                x0, y0 = bbox[0] * zoom, bbox[1] * zoom
                x1, y1 = bbox[2] * zoom, bbox[3] * zoom
                cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
                if not dentro_crop(cx, cy):
                    continue
                ax0, ay0 = ajustar(x0, y0)
                cotas_texto.append({
                    'texto': texto,
                    'x': round(ax0), 'y': round(ay0),
                    'w': round(x1 - x0), 'h': round(y1 - y0),
                })

    # ── Trazos vectoriales cortos: muros ya cubiertos por OpenCV, nos
    #    interesan los candidatos a simbolo (arco de puerta, linea de ventana)
    trazos = []
    n_descartados_largos = 0
    max_largo_px = (max_largo_trazo_m / mpx) if mpx else float('inf')

    # ── Lineas discontinuas: sin filtro de largo, para borrarlas del raster
    lineas_discontinuas = []

    # DIAGNOSTICO 2026-07-24: el fix de borrar 'trazos' cortos del raster
    # (por largo <3m) causo una regresion grave -- segmentos reales de muro
    # perimetral en esquinas/quiebres tambien miden <3m y se borraron,
    # punzando agujeros en el muro exterior y fusionando exterior+interior
    # en un solo "recinto" de ~140-155 m2 (ver roadmap P1). Antes de re-
    # intentar el fix con un criterio mejor (ancho de linea en vez de largo),
    # se recolectan datos reales en vez de adivinar de nuevo a ciegas.
    _anchos_cortos = []
    _anchos_largos = []
    _dashes_muestra = set()

    for path in pdf_page.get_drawings():
        ancho_linea = path.get('width') or 0
        if len(_dashes_muestra) < 15:
            _dashes_muestra.add(repr(path.get('dashes')))
        es_discontinuo = _es_trazo_discontinuo(path)

        if es_discontinuo:
            pts_path = []
            for item in path.get('items', []):
                op = item[0]
                if op == 'l':
                    pts_path.extend([item[1], item[2]])
                elif op == 'c':
                    pts_path.extend(item[1:5])
                elif op == 're':
                    r = item[1]
                    pts_path.extend([r.tl, r.tr, r.br, r.bl])
                elif op == 'qu':
                    q = item[1]
                    pts_path.extend([q.ul, q.ur, q.lr, q.ll])
            if len(pts_path) >= 2:
                pts_px = [to_px(p) for p in pts_path]
                if any(dentro_crop(x, y) for x, y in pts_px):
                    pts_ajustados = [ajustar(x, y) for x, y in pts_px]
                    lineas_discontinuas.append({
                        'puntos': [(round(x), round(y)) for x, y in pts_ajustados],
                        'ancho_linea': round(ancho_linea, 2),
                    })
            # una linea discontinua no es candidata a simbolo — no sigue al
            # bloque de 'trazos' cortos de abajo
            continue

        for item in path.get('items', []):
            op = item[0]
            puntos_px = []
            if op == 'l':      # linea: 2 puntos
                puntos_px = [to_px(item[1]), to_px(item[2])]
            elif op == 'c':    # curva bezier (tipico en arcos de puerta)
                puntos_px = [to_px(p) for p in item[1:5]]
            elif op == 're':   # rectangulo
                r = item[1]
                puntos_px = [to_px(r.tl), to_px(r.tr), to_px(r.br), to_px(r.bl)]
            elif op == 'qu':   # quad
                q = item[1]
                puntos_px = [to_px(q.ul), to_px(q.ur), to_px(q.lr), to_px(q.ll)]
            else:
                continue

            cx = sum(p[0] for p in puntos_px) / len(puntos_px)
            cy = sum(p[1] for p in puntos_px) / len(puntos_px)
            if not dentro_crop(cx, cy):
                continue

            # Descartar trazos largos: son muros/limites, ya cubiertos por
            # OpenCV. Nos interesan los cortos: candidatos a simbolo puntual.
            xs = [p[0] for p in puntos_px]; ys = [p[1] for p in puntos_px]
            largo_aprox = max(max(xs) - min(xs), max(ys) - min(ys))
            if largo_aprox > max_largo_px:
                n_descartados_largos += 1
                _anchos_largos.append(round(ancho_linea, 3))
                continue

            _anchos_cortos.append(round(ancho_linea, 3))
            puntos_ajustados = [ajustar(x, y) for x, y in puntos_px]
            trazos.append({
                'tipo': op,
                'puntos': [(round(x), round(y)) for x, y in puntos_ajustados],
                'ancho_linea': round(ancho_linea, 2),
            })

    def _stats(vals):
        if not vals:
            return None
        vals_ord = sorted(vals)
        n = len(vals_ord)
        return {
            'n': n, 'min': vals_ord[0], 'max': vals_ord[-1],
            'mediana': vals_ord[n // 2],
            'promedio': round(sum(vals_ord) / n, 3),
        }

    return {
        'cotas_texto': cotas_texto,
        'trazos': trazos,
        'lineas_discontinuas': lineas_discontinuas,
        'diagnostico_anchos_trazos_cortos': _stats(_anchos_cortos),
        'diagnostico_anchos_muros_largos': _stats(_anchos_largos),
        'diagnostico_dashes_muestra': list(_dashes_muestra),
        'n_texto': len(cotas_texto),
        'n_trazos': len(trazos),
        'n_trazos_descartados_largos': n_descartados_largos,
        'n_lineas_discontinuas': len(lineas_discontinuas),
    }

resultados_paginas = []
viz_pages          = []

entries = [(e[0], e[1], e[2] if len(e) > 2 else None) for e in PAGINAS_Y_ESCALAS]

# Precalcular cuántas veces aparece cada página (para nombres de archivo únicos)
page_count = {}
for pag, _, _ in entries:
    page_count[pag] = page_count.get(pag, 0) + 1
page_idx_so_far = {}

for (PAGINA_PLANTA, ESCALA_MANUAL, crop) in entries:
    print(f'\n{"="*56}')
    print(f'  Página {PAGINA_PLANTA}  —  escala {ESCALA_MANUAL}')
    if crop:
        print(f'  Recorte: ({crop[0]:.0%},{crop[1]:.0%}) → ({crop[2]:.0%},{crop[3]:.0%})')
    print(f'{"="*56}')

    if PAGINA_PLANTA < 1 or PAGINA_PLANTA > len(paginas):
        print(f'  ⚠ Página {PAGINA_PLANTA} fuera de rango (PDF tiene {len(paginas)} páginas). Saltando.')
        continue

    # Índice único por entrada (resuelve el caso de 2 crops de la misma página)
    entry_idx = len(resultados_paginas)
    page_idx_so_far[PAGINA_PLANTA] = page_idx_so_far.get(PAGINA_PLANTA, 0) + 1
    sub_idx = page_idx_so_far[PAGINA_PLANTA]
    fname_tag = (f'pag{PAGINA_PLANTA}-{sub_idx}'
                 if page_count[PAGINA_PLANTA] > 1
                 else f'pag{PAGINA_PLANTA}')

    plano_full = paginas[PAGINA_PLANTA - 1]
    h_f, w_f   = plano_full.shape[:2]

    # Aplicar recorte si está definido
    if crop:
        x1f, y1f, x2f, y2f = crop
        x1 = int(x1f * w_f); y1 = int(y1f * h_f)
        x2 = int(x2f * w_f); y2 = int(y2f * h_f)
        plano = plano_full[y1:y2, x1:x2].copy()
    else:
        plano = plano_full

    h, w  = plano.shape[:2]
    scale_ratio = int(ESCALA_MANUAL.split(':')[1])
    MPX   = 0.0254 * scale_ratio / DPI
    M2_PX = MPX ** 2
    print(f'  {w}x{h} px analizados  |  {MPX:.5f} m/px  |  1m = {int(1/MPX):,} px')

    # ── 1. Claude Vision ────────────────────────────────────
    print('  → Claude Vision...')
    _, buf_orig = cv2.imencode('.png', cv2.cvtColor(plano, cv2.COLOR_RGB2BGR))
    img_b64_orig = base64.standard_b64encode(buf_orig.tobytes()).decode()

    # Segunda imagen: version con contraste/nitidez mejorada, como referencia
    # adicional para que Claude vuelva a mirar puertas/ventanas dificiles de ver.
    plano_mejorado = mejorar_contraste_nitidez(plano)
    _, buf_mejor = cv2.imencode('.png', cv2.cvtColor(plano_mejorado, cv2.COLOR_RGB2BGR))
    img_b64_mejor = base64.standard_b64encode(buf_mejor.tobytes()).decode()

    PROMPT = (
        f'Eres revisor DOM experto en OGUC, LGUC y DDU (Chile). '
        f'Analiza este plano arquitectonico a escala {ESCALA_MANUAL}.\n'
        'Te doy DOS imagenes del mismo plano: la primera es la imagen original a color, '
        'la segunda es una version con contraste y nitidez realzados (en blanco y negro) — '
        'usa la segunda especificamente para buscar puertas y ventanas que sean dificiles '
        'de distinguir en la primera por el relleno de color de los recintos.\n'
        'Devuelve SOLO JSON puro sin markdown ni texto extra:\n'
        '{"tipo_plano":"planta|corte|elevacion|detalle|otro",'
        '"uso_del_proyecto":"restaurante|vivienda|oficina|comercio|equipamiento|otro",'
        '"nivel":"descripcion o null",'
        '"recintos":[{"nombre":"...","tipo":"sala|cocina|bano|bodega|pasillo|terraza|bar|oficina|rampa|escalera|otro",'
        '"etiqueta_en_plano":"texto exacto o null","area_estimada_m2":null,"ancho_estimado_m":null,'
        '"cx_relativo":0.5,"cy_relativo":0.5,"cumple_oguc":true,"observacion":"o null"}],'
        '"elementos_detectados":{"puertas":0,"ventanas":0,"escaleras":0,"rampas":0,"salidas_emergencia":0},'
        '"incumplimientos_oguc":[{"articulo":"","descripcion":"","gravedad":"ALTA|MEDIA|BAJA",'
        '"recinto_afectado":"","medida_requerida":"","medida_detectada":""}],'
        '"documentos_que_faltan":[],"resumen_ejecutivo":""}\n'
        'cx_relativo/cy_relativo: centroide del recinto como fraccion del ancho/alto '
        '(0.0=izquierda/arriba, 1.0=derecha/abajo).'
    )

    analisis = {
        'tipo_plano': '?', 'uso_del_proyecto': '?', 'nivel': '?',
        'recintos': [], 'elementos_detectados': {},
        'incumplimientos_oguc': [], 'documentos_que_faltan': [],
        'resumen_ejecutivo': 'Sin analisis semantico'
    }
    try:
        resp = requests.post(
            WORKER_URL,
            json={'messages': [{'role': 'user', 'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64_orig}},
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64_mejor}},
                {'type': 'text', 'text': PROMPT}
            ]}]},
            timeout=180, stream=True
        )
        resp.raise_for_status()
        raw_text = ''
        for line in resp.iter_lines():
            if not line:
                continue
            line = line.decode('utf-8') if isinstance(line, bytes) else line
            if not line.startswith('data: '):
                continue
            payload = line[6:].strip()
            if not payload or payload == '[DONE]':
                continue
            try:
                evt = json.loads(payload)
                if (evt.get('type') == 'content_block_delta' and
                        evt.get('delta', {}).get('type') == 'text_delta'):
                    raw_text += evt['delta']['text']
            except:
                pass
        raw_text = raw_text.replace('```json', '').replace('```', '').strip()
        m = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if m:
            analisis  = json.loads(m.group())
            r_count   = len(analisis.get('recintos', []))
            inc_count = len(analisis.get('incumplimientos_oguc', []))
            print(f'  ✓ Claude: {r_count} recintos, {inc_count} incumplimientos')
        else:
            print('  ⚠ Claude: sin JSON en respuesta')
    except Exception as e:
        print(f'  ⚠ Error Claude: {e}')

    # ── 2. Extracción de datos vectoriales del PDF ──────────
    # NOTA 2026-07-23: se movio ANTES de OpenCV (antes iba despues) porque
    # OpenCV ahora necesita cotas_texto y lineas_discontinuas para limpiar
    # el raster antes de binarizar. Requiere que la Celda 2 haya confirmado
    # PDF vectorizado. `doc` sigue vivo en el kernel desde la Celda 2.
    print('  → Extracción vectorial...')
    pdf_page_actual = doc[PAGINA_PLANTA - 1]
    crop_px = (x1, y1, x2, y2) if crop else None
    datos_vectoriales = extraer_datos_vectoriales(pdf_page_actual, ZOOM, MPX, crop_px)
    print(f'  ✓ Vectorial: {datos_vectoriales["n_texto"]} textos (cotas/nombres), '
          f'{datos_vectoriales["n_trazos"]} trazos candidatos a símbolo '
          f'({datos_vectoriales["n_trazos_descartados_largos"]} descartados por largos — son muros), '
          f'{datos_vectoriales["n_lineas_discontinuas"]} líneas discontinuas (deslinde/eje/línea de edificación)')

    # ── 3. OpenCV — extracción geométrica ───────────────────
    print('  → OpenCV...')
    gray = cv2.cvtColor(plano, cv2.COLOR_RGB2GRAY)

    # FIX 2026-07-23 (a): borrar texto (cotas, nombres de recintos) usando
    # las posiciones exactas del vector — el texto NO debe limitar el area
    # de un recinto. Antes "0.8" o "Cocina" se trataban como si fueran parte
    # del muro porque adaptiveThreshold no distingue texto de linea.
    PADDING_TEXTO_PX = 3
    n_texto_borrado = 0
    for t in datos_vectoriales['cotas_texto']:
        tx0 = max(0, t['x'] - PADDING_TEXTO_PX)
        ty0 = max(0, t['y'] - PADDING_TEXTO_PX)
        tx1 = min(w, t['x'] + t['w'] + PADDING_TEXTO_PX)
        ty1 = min(h, t['y'] + t['h'] + PADDING_TEXTO_PX)
        if tx1 > tx0 and ty1 > ty0:
            gray[ty0:ty1, tx0:tx1] = 255
            n_texto_borrado += 1

    # FIX 2026-07-23 (b): borrar líneas discontinuas (deslinde, línea de
    # edificación, ejes) — son referencias, no muros, y no deben separar
    # recintos en dos. Se identifican por el patrón de guiones real del PDF
    # (path['dashes']), no por una heurística de grosor de píxeles.
    n_lineas_borradas = 0
    for ld in datos_vectoriales['lineas_discontinuas']:
        grosor_borrado = max(6, int(ld['ancho_linea'] * ZOOM) + 6)  # margen anti-aliasing
        pts = ld['puntos']
        for i in range(len(pts) - 1):
            cv2.line(gray, pts[i], pts[i + 1], 255, thickness=grosor_borrado)
        n_lineas_borradas += 1

    # FIX 2026-07-23 (c) — REVERTIDO 2026-07-24: causaba una regresion grave.
    # Borrar 'trazos' cortos (<3m, mismo filtro que ya existia para candidatos
    # a simbolo) por LARGO solamente resulto inseguro: segmentos reales de
    # muro perimetral en esquinas/quiebres del contorno del edificio tambien
    # miden <3m, y se borraban igual que un icono de artefacto o un arco de
    # puerta — punzando agujeros en el muro exterior real. Resultado medido:
    # el recinto mas grande de cada plano paso a medir 141-155 m2, mezclando
    # el exterior completo con fragmentos del interior en un solo blob falso
    # (ver capturas del usuario + verif_recintos_pag2-1/2.png, 2026-07-24).
    # NO se vuelve a activar el borrado hasta tener un criterio mejor que
    # largo (ver diagnostico de anchos abajo — la idea es usar ancho_linea,
    # ya que un muro real casi seguro se dibuja mas grueso que un icono o
    # arco de puerta, pero hace falta el dato real del PDF antes de fijar
    # un umbral, no adivinar una segunda vez a ciegas).
    n_trazos_borrados = 0  # deliberadamente 0 mientras el fix esta revertido

    print(f'  ✓ Limpieza pre-umbral: {n_texto_borrado} textos, {n_lineas_borradas} líneas discontinuas borrados '
          f'(borrado de trazos cortos DESACTIVADO — ver nota de regresión en el código)')

    dv = datos_vectoriales
    print(f'  📊 DIAGNÓSTICO para diseñar el fix correcto (no borra nada, solo informa):')
    print(f'     ancho_linea trazos CORTOS (<3m, candidatos a símbolo): {dv["diagnostico_anchos_trazos_cortos"]}')
    print(f'     ancho_linea trazos LARGOS (>3m, asumidos muro):        {dv["diagnostico_anchos_muros_largos"]}')
    print(f'     muestra cruda de path["dashes"] (para depurar por qué n_lineas_discontinuas={dv["n_lineas_discontinuas"]}): '
          f'{dv["diagnostico_dashes_muestra"]}')

    binary_inv = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, blockSize=21, C=4)
    k_close    = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    muros      = cv2.dilate(binary_inv, k_close, iterations=2)
    k_open     = cv2.getStructuringElement(cv2.MORPH_RECT, (10, 10))
    limpios    = cv2.morphologyEx(cv2.bitwise_not(muros), cv2.MORPH_OPEN, k_open)

    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        limpios, connectivity=8)

    MIN_PX2 = int(0.5 / M2_PX)
    recintos_geo = []
    for idx in range(1, n_labels):
        area_px = int(stats[idx, cv2.CC_STAT_AREA])
        if area_px < MIN_PX2:
            continue
        area_m2 = round(area_px * M2_PX, 2)
        cx_abs  = int(centroids[idx][0])
        cy_abs  = int(centroids[idx][1])
        mask    = (labels == idx).astype(np.uint8) * 255
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        ancho_m = largo_m = None
        bbox = None
        if cnts:
            cnt  = max(cnts, key=cv2.contourArea)
            rect = cv2.minAreaRect(cnt)
            dims = sorted(rect[1])
            ancho_m = round(dims[0] * MPX, 2)
            largo_m = round(dims[1] * MPX, 2)
            bx, by, bw, bh = cv2.boundingRect(cnt)
            bbox = {'x': int(bx), 'y': int(by), 'w': int(bw), 'h': int(bh)}
        recintos_geo.append({
            'id'         : f'E{len(recintos_geo)+1:02d}',
            'label'      : idx,
            'area_px'    : area_px,
            'area_m2'    : area_m2,
            'ancho_min_m': ancho_m,
            'largo_max_m': largo_m,
            'cx'         : cx_abs,
            'cy'         : cy_abs,
            'cx_rel'     : round(cx_abs / w, 3),
            'cy_rel'     : round(cy_abs / h, 3),
            'bbox'       : bbox,
        })
    recintos_geo.sort(key=lambda r: r['area_m2'], reverse=True)
    print(f'  ✓ OpenCV: {len(recintos_geo)} espacios >= 0.5 m²')

    # ── 4. Cruce semántica + geometría ──────────────────────
    recintos_claude = analisis.get('recintos', [])
    usados = set()

    def dist_rel(rg, rc):
        return math.sqrt((rg['cx_rel'] - rc.get('cx_relativo', 0.5))**2 +
                         (rg['cy_rel'] - rc.get('cy_relativo', 0.5))**2)

    def mejor_match(rg):
        cands = [(dist_rel(rg, rc), j, rc)
                 for j, rc in enumerate(recintos_claude) if j not in usados]
        if not cands:
            return None
        cands.sort(key=lambda x: x[0])
        d, j, rc = cands[0]
        if d < 0.25:
            usados.add(j)
            return rc
        return None

    tabla = []
    incumplimientos_geo = []

    for rg in recintos_geo:
        rc     = mejor_match(rg)
        # FIX 2026-07-23 (c): un recinto sin match de Claude Vision ya NO se
        # nombra en silencio ("Espacio E##") — se marca explicitamente para
        # que el arquitecto lo confirme (via la interfaz de validacion grafica
        # cuando exista; por ahora, print de advertencia + campo dedicado).
        sin_nombre = rc is None
        nombre = rc['nombre'] if rc else f'Espacio {rg["id"]} (SIN NOMBRE - confirmar con arquitecto)'
        tipo   = (rc['tipo'] if rc else 'otro').lower().split('/')[0].strip()
        area   = rg['area_m2']
        ancho  = rg['ancho_min_m']

        area_min, ancho_min, ref = OGUC_REGLAS.get(tipo, (None, None, None))
        area_ok = ancho_ok = None

        if area_min and area < area_min:
            area_ok = False
            incumplimientos_geo.append({
                'tipo': 'area', 'pagina': PAGINA_PLANTA,
                'recinto': nombre, 'id': rg['id'],
                'medido': area, 'minimo': area_min,
                'deficit': round(area_min - area, 2), 'ref': ref
            })
        if ancho_min and ancho is not None and ancho < ancho_min:
            ancho_ok = False
            incumplimientos_geo.append({
                'tipo': 'ancho', 'pagina': PAGINA_PLANTA,
                'recinto': nombre, 'id': rg['id'],
                'medido': ancho, 'minimo': ancho_min,
                'deficit': round(ancho_min - ancho, 2), 'ref': ref
            })

        tabla.append({
            'id'                   : rg['id'],
            'nombre'               : nombre,
            'tipo'                 : tipo,
            'pagina'               : PAGINA_PLANTA,
            'area_m2'              : area,
            'ancho_min_m'          : ancho,
            'largo_max_m'          : rg.get('largo_max_m'),
            'cumple_geo'           : (area_ok is not False) and (ancho_ok is not False),
            'cx_rel'               : rg['cx_rel'],
            'cy_rel'               : rg['cy_rel'],
            'bbox'                 : rg.get('bbox'),
            'sin_nombre_confirmar' : sin_nombre,
        })

    total   = round(sum(f['area_m2'] for f in tabla), 1)
    matched = sum(1 for f in tabla if not f['sin_nombre_confirmar'])
    print(f'  ✓ Cruce: {matched}/{len(tabla)} con nombre | {len(incumplimientos_geo)} incumpl. geo')

    sin_nombre_ids = [f['id'] for f in tabla if f['sin_nombre_confirmar']]
    if sin_nombre_ids:
        print(f'  ⚠ {len(sin_nombre_ids)} espacio(s) SIN NOMBRE — requieren que el arquitecto confirme qué son: {", ".join(sin_nombre_ids)}')

    resultados_paginas.append({
        'entry_idx'             : entry_idx,
        'fname_tag'             : fname_tag,
        'pagina'                : PAGINA_PLANTA,
        'escala'                : ESCALA_MANUAL,
        'crop'                  : list(crop) if crop else None,
        'analisis_semantico'    : analisis,
        'mediciones_geometricas': tabla,
        'incumplimientos_geo'   : incumplimientos_geo,
        'datos_vectoriales'     : datos_vectoriales,
        'total_area_m2'         : total,
        'imagen_w_px'           : w,
        'imagen_h_px'           : h,
        'mpp'                   : MPX,
    })
    viz_pages.append({
        'entry_idx'   : entry_idx,
        'fname_tag'   : fname_tag,
        'pagina'      : PAGINA_PLANTA,
        'escala'      : ESCALA_MANUAL,
        'plano'       : plano,
        'labels'      : labels,
        'recintos_geo': recintos_geo,
        'w': w, 'h': h,
    })

print(f'\n{"="*56}')
total_inc = sum(len(p['incumplimientos_geo']) for p in resultados_paginas)
print(f'✓ Procesadas {len(resultados_paginas)} / {len(PAGINAS_Y_ESCALAS)} páginas')
print(f'  Incumplimientos geométricos totales: {total_inc}')


In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 4b — Cargar Grounding DINO + SAM 2  [GPU requerida]
#
# Ejecutar UNA VEZ por sesión de Colab.
# No es necesario re-ejecutar si vuelves a correr Celda 4.
# ══════════════════════════════════════════════════════════

# 1. Verificar GPU — obligatorio
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n⛔  GPU no disponible.\n"
        "    Ve a: Runtime → Change runtime type → Hardware accelerator: T4 GPU\n"
        "    Luego reinicia la sesión y ejecuta desde Celda 1.\n"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {gpu_name}  ({vram_gb:.1f} GB VRAM)")

# 2. Instalar librerías
print("\nInstalando Grounding DINO + SAM 2 (~2–3 min la primera vez)...")
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.45.0",
     "git+https://github.com/facebookresearch/sam2.git"],
    check=True
)
print("✓ Librerías instaladas")

# 3. Cargar Grounding DINO (HuggingFace)
# size override: DINO por defecto procesa ~800px — subimos a 2048px para
# detectar elementos pequeños (puertas, ventanas) en planos de alta resolución
print("\nCargando Grounding DINO base (~700 MB) con resolución 2048px...")
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

DINO_ID        = "IDEA-Research/grounding-dino-base"
dino_processor = AutoProcessor.from_pretrained(
    DINO_ID,
    size={"shortest_edge": 2048, "longest_edge": 2048}
)
dino_model     = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID).to("cuda")
dino_model.eval()
print("✓ Grounding DINO listo (resolución máx. 2048px)")

# 4. Cargar SAM 2 — hiera-small (~185 MB)
print("\nCargando SAM 2 hiera-small (~185 MB)...")
from sam2.build_sam import build_sam2_hf
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_predictor = SAM2ImagePredictor(build_sam2_hf("facebook/sam2.1-hiera-small"))
print("✓ SAM 2 listo")

vram_usada = torch.cuda.memory_allocated() / 1e9
print(f"\nVRAM usada: {vram_usada:.1f} / {vram_gb:.1f} GB  ({vram_usada/vram_gb*100:.0f}%)")
print("\n✓ Modelos listos. Ejecuta la Celda 4c.")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 4c — Detección de elementos con DINO + SAM 2
#
# Detecta puertas, ventanas, escaleras, rampas, columnas
# en cada página ya procesada por OpenCV (Celda 4).
# Complementa la detección de recintos — no la reemplaza.
#
# Nota: DINO fue entrenado en fotografías. En planos 2D
# detecta símbolos reconocibles (puertas, ventanas) con
# ~60–75% de recall. Las mediciones son orientativas —
# confirmar siempre con cotas del plano.
# ══════════════════════════════════════════════════════════
from PIL import Image
import numpy as np, torch, cv2

# Prompt en inglés — DINO es más preciso en inglés para símbolos arquitectónicos
# FIX 2026-07-20: Grounding DINO agrupa frases por punto ".", y la convención
# oficial del modelo (README / model card IDEA-Research) es que el prompt
# TERMINE en punto. Sin el punto final, la última frase ("emergency exit")
# puede no agruparse bien en el post-proceso. Antes de este fix: sin punto final.
PROMPT         = "door . window . staircase . ramp . column . emergency exit ."
BOX_THRESHOLD  = 0.20   # ajustar: bajar a 0.15 si detecta poco; subir a 0.30 si hay falsos positivos
TEXT_THRESHOLD = 0.15

# DIAGNÓSTICO 2026-07-20: en las 5 corridas guardadas hasta ahora, DINO detectó
# en total 1 elemento (13% de confianza) en las 5 combinadas — muy por debajo
# del ~60-75% de recall documentado arriba. El fix del punto final puede ayudar,
# pero antes de asumir que fue la única causa, correr al menos una vez con
# DEBUG_THRESHOLD_MINIMO=True para ver si aparece CUALQUIER señal por debajo
# del threshold normal. Si con esto tampoco aparece nada, el problema probable-
# mente no es de threshold/prompt sino de brecha de dominio (DINO fue entrenado
# en fotografías, no en símbolos de líneas de plano CAD) — en ese caso lo más
# útil de este experimento es documentarlo y apoyarse más en el conteo semántico
# de Claude Vision (ya existe como fallback automático más abajo).
DEBUG_THRESHOLD_MINIMO = False
if DEBUG_THRESHOLD_MINIMO:
    BOX_THRESHOLD  = 0.05
    TEXT_THRESHOLD = 0.05
    print("⚠ MODO DIAGNÓSTICO: threshold bajado a 0.05 — esperar más falsos positivos, es solo para ver si hay señal real")

TIPO_ES = {
    "door"          : "puerta",
    "window"        : "ventana",
    "staircase"     : "escalera",
    "ramp"          : "rampa",
    "column"        : "columna",
    "emergency exit": "salida_emergencia",
}

# Dimensión máxima razonable por tipo (m) — superar esto = falso positivo
MAX_ANCHO_M = {
    "puerta"           : 2.5,   # ninguna puerta supera 2.5 m de hoja
    "ventana"          : 4.0,   # ventana corrida máx ~4 m
    "escalera"         : 6.0,   # escalera máx ~6 m de ancho
    "rampa"            : 4.0,   # rampa máx ~4 m de ancho
    "columna"          : 1.5,   # columna máx ~1.5 m
    "salida_emergencia": 3.0,
}
MIN_CONFIANZA = 0.15  # descartar detecciones con score < 15%

# Colores por tipo para visualización
COLORES_DINO = {
    "puerta"           : (30,  144, 255),
    "ventana"          : (0,   206, 209),
    "escalera"         : (255, 140,   0),
    "rampa"            : (148,   0, 211),
    "columna"          : (220,  20,  60),
    "salida_emergencia": (34,  139,  34),
}

print(f"Procesando {len(viz_pages)} página(s) con Grounding DINO + SAM 2...\n")

for vz in viz_pages:
    pag   = vz['pagina']
    plano = vz['plano']
    h, w  = plano.shape[:2]

    # Usar entry_idx para match directo — evita el bug cuando hay 2 crops de la misma página
    res_pag = resultados_paginas[vz['entry_idx']]
    MPX = res_pag['mpp']

    print(f"  Página {pag} [{vz['fname_tag']}]  |  {res_pag['escala']}  |  {w}×{h} px")

    # ── Grounding DINO ─────────────────────────────────────
    pil_img = Image.fromarray(plano)
    inputs  = dino_processor(
        images=pil_img, text=PROMPT, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = dino_model(**inputs)

    detecciones = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold      = BOX_THRESHOLD,
        text_threshold = TEXT_THRESHOLD,
        target_sizes   = [(h, w)],
    )[0]

    boxes  = detecciones["boxes"].cpu().numpy()
    labels = detecciones["labels"]
    scores = detecciones["scores"].cpu().numpy()

    # Auto-retry con threshold reducido si no hay detecciones
    if len(boxes) == 0:
        retry_thr      = max(BOX_THRESHOLD * 0.65, 0.12)
        retry_text_thr = max(TEXT_THRESHOLD * 0.65, 0.10)
        print(f"    ⚠ Sin detecciones (thr={BOX_THRESHOLD:.2f}) — reintentando con thr={retry_thr:.2f}...")
        retry_det = dino_processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids,
            threshold      = retry_thr,
            text_threshold = retry_text_thr,
            target_sizes   = [(h, w)],
        )[0]
        boxes  = retry_det["boxes"].cpu().numpy()
        labels = retry_det["labels"]
        scores = retry_det["scores"].cpu().numpy()
        if len(boxes) == 0:
            print(f"    ⚠ Sin detecciones incluso con threshold reducido — usando conteo semántico como respaldo\n")
            res_pag["elementos_dino"] = []
            vz["elementos_dino"]      = []
            continue
        print(f"    → {len(boxes)} candidato(s) con threshold reducido — aplicando filtro de calidad")

    # ── SAM 2 — máscara precisa por cada detección ─────────
    elementos = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        sam2_predictor.set_image(plano)
        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
            ancho_m = round((x2 - x1) * MPX, 3)
            alto_m  = round((y2 - y1) * MPX, 3)

            masks, _, _ = sam2_predictor.predict(
                box=np.array([x1, y1, x2, y2])[None],
                multimask_output=False,
            )
            area_mascara = round(float(masks[0].sum()) * MPX ** 2, 3)

            tipo_es = TIPO_ES.get(label, label)
            elementos.append({
                "tipo"           : tipo_es,
                "tipo_en"        : label,
                "confianza"      : round(float(score), 3),
                "bbox_px"        : [round(x1), round(y1), round(x2), round(y2)],
                "ancho_m"        : ancho_m,
                "alto_m"         : alto_m,
                "area_mascara_m2": area_mascara,
                "cx_rel"         : round((x1 + x2) / 2 / w, 3),
                "cy_rel"         : round((y1 + y2) / 2 / h, 3),
            })

    # ── Filtro de calidad — eliminar falsos positivos ───────
    n_bruto = len(elementos)
    elementos = [
        e for e in elementos
        if e["confianza"] >= MIN_CONFIANZA
        and e["ancho_m"] <= MAX_ANCHO_M.get(e["tipo"], 10.0)
    ]
    n_filtrados = n_bruto - len(elementos)
    if n_filtrados > 0:
        print(f"    ⚠ {n_filtrados} detección(es) descartada(s) (confianza < {MIN_CONFIANZA} o dimensión fuera de rango)")

    if not elementos:
        print(f"    ⚠ Sin detecciones válidas tras filtro — usando conteo semántico como respaldo\n")
        res_pag["elementos_dino"] = []
        vz["elementos_dino"]      = []
        continue

    res_pag["elementos_dino"] = elementos
    vz["elementos_dino"]      = elementos

    # Resumen
    conteo = {}
    for e in elementos:
        conteo[e["tipo"]] = conteo.get(e["tipo"], 0) + 1
    print(f"    ✓ {len(elementos)} detectado(s): {conteo}")

    # Alerta puertas angostas (OGUC Art. 4.2.2 — mín 0.90 m)
    angostas = [e for e in elementos if e["tipo"] == "puerta" and e["ancho_m"] < 0.90]
    if angostas:
        print(f"    ⚠ {len(angostas)} puerta(s) < 0.90 m — verificar con cota en plano (OGUC Art. 4.2.2)")

    # Alerta escaleras angostas (OGUC Art. 4.2.4 — mín 1.20 m)
    esc_angostas = [e for e in elementos if e["tipo"] == "escalera" and e["ancho_m"] < 1.20]
    if esc_angostas:
        print(f"    ⚠ {len(esc_angostas)} escalera(s) < 1.20 m — verificar (OGUC Art. 4.2.4)")

    print()

# Totales globales
puertas_n   = sum(sum(1 for e in r.get("elementos_dino",[]) if e["tipo"]=="puerta")    for r in resultados_paginas)
ventanas_n  = sum(sum(1 for e in r.get("elementos_dino",[]) if e["tipo"]=="ventana")   for r in resultados_paginas)
escaleras_n = sum(sum(1 for e in r.get("elementos_dino",[]) if e["tipo"]=="escalera")  for r in resultados_paginas)
rampas_n    = sum(sum(1 for e in r.get("elementos_dino",[]) if e["tipo"]=="rampa")     for r in resultados_paginas)

print(f"✓ Detección completa — totales:")
print(f"  Puertas   : {puertas_n}")
print(f"  Ventanas  : {ventanas_n}")
print(f"  Escaleras : {escaleras_n}")
print(f"  Rampas    : {rampas_n}")
print("\nEjecuta Celda 5 para ver la visualización con detecciones.")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 5 — Visualización: OpenCV + detecciones DINO
# ══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import hsv_to_rgb
import cv2, numpy as np

def color_recinto(i, n):
    rgb = hsv_to_rgb([(i / max(n, 1)) * 0.85, 0.60, 0.95])
    return tuple(int(c * 255) for c in rgb)

# Colores DINO por tipo (BGR para cv2)
COLORES_DINO_BGR = {
    "puerta"           : (255, 144,  30),
    "ventana"          : (209, 206,   0),
    "escalera"         : (  0, 140, 255),
    "rampa"            : (211,   0, 148),
    "columna"          : ( 60,  20, 220),
    "salida_emergencia": ( 34, 139,  34),
}

n_rows = len(viz_pages)
fig, axes = plt.subplots(n_rows, 2, figsize=(22, 11 * n_rows))
if n_rows == 1:
    axes = [axes]

for row, vz in enumerate(viz_pages):
    plano_v    = vz['plano']
    labels_v   = vz['labels']
    recintos_v = vz['recintos_geo']
    ESCALA_V   = vz['escala']
    PAGINA_V   = vz['pagina']
    fname_tag  = vz['fname_tag']
    elementos_dino = vz.get('elementos_dino', [])
    n = len(recintos_v)

    # ── Capa OpenCV (recintos coloreados) ──────────────────
    anotado = plano_v.copy()
    for i, r in enumerate(recintos_v):
        cr, cg, cb = color_recinto(i, n)
        mask = (labels_v == r['label']).astype(np.uint8) * 255
        if mask.any():
            pix = anotado[mask > 0]
            anotado[mask > 0] = [
                int(pix[:, 0].mean() * 0.45 + cr * 0.55),
                int(pix[:, 1].mean() * 0.45 + cg * 0.55),
                int(pix[:, 2].mean() * 0.45 + cb * 0.55),
            ]
        cx, cy = r['cx'], r['cy']
        font   = cv2.FONT_HERSHEY_SIMPLEX
        txts   = [r['id'], f"{r['area_m2']} m2"]
        if r['ancho_min_m']:
            txts.append(f"a:{r['ancho_min_m']}m")
        for ti, txt in enumerate(txts):
            yy = cy - 16 + ti * 18
            for dx, dy in [(-1, -1), (1, -1), (-1, 1), (1, 1)]:
                cv2.putText(anotado, txt, (cx + dx - 20, yy + dy), font, 0.55, (0, 0, 0), 2)
            col = (255, 255, 255) if ti == 0 else (230, 230, 60)
            cv2.putText(anotado, txt, (cx - 20, yy), font, 0.55, col, 1)

    # ── Capa DINO (bboxes de elementos) ────────────────────
    for e in elementos_dino:
        x1, y1, x2, y2 = e['bbox_px']
        color_bgr = COLORES_DINO_BGR.get(e['tipo'], (128, 128, 128))
        anotado_bgr = cv2.cvtColor(anotado, cv2.COLOR_RGB2BGR)
        cv2.rectangle(anotado_bgr, (x1, y1), (x2, y2), color_bgr, 3)
        label_txt = f"{e['tipo']} {e['ancho_m']}m ({e['confianza']:.0%})"
        (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(anotado_bgr, (x1, y1 - th - 6), (x1 + tw + 4, y1), color_bgr, -1)
        cv2.putText(anotado_bgr, label_txt,
                    (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        anotado = cv2.cvtColor(anotado_bgr, cv2.COLOR_BGR2RGB)

    # PNG individual: {BASENAME}_{fname_tag}.png
    # Ejemplo: archicheck_geometrico_pdv_30jun_1729_pag2-1.png
    fname_pag = f'{BASENAME}_{fname_tag}.png'
    cv2.imwrite(fname_pag, cv2.cvtColor(anotado, cv2.COLOR_RGB2BGR))
    print(f'  ✓ PNG guardado: {fname_pag}')

    ax0, ax1 = axes[row][0], axes[row][1]
    ax0.imshow(plano_v)
    ax0.set_title(f'Página {PAGINA_V} [{fname_tag}] — original', fontsize=12, fontweight='bold')
    ax0.axis('off')

    ax1.imshow(anotado)
    dino_info = f" | DINO: {len(elementos_dino)} elementos" if elementos_dino else ""
    ax1.set_title(
        f'Página {PAGINA_V} [{fname_tag}] — OpenCV ({n} espacios){dino_info} | {ESCALA_V}',
        fontsize=12, fontweight='bold'
    )
    ax1.axis('off')

    patches_cv = [
        mpatches.Patch(
            color=[c / 255 for c in color_recinto(i, n)],
            label=f"{r['id']}: {r['area_m2']} m²" + (f" · a:{r['ancho_min_m']}m" if r['ancho_min_m'] else '')
        )
        for i, r in enumerate(recintos_v[:12])
    ]
    patches_dino = [
        mpatches.Patch(facecolor=[c/255 for c in (30,144,255)],  label='Puerta (DINO)'),
        mpatches.Patch(facecolor=[c/255 for c in (0,206,209)],   label='Ventana (DINO)'),
        mpatches.Patch(facecolor=[c/255 for c in (255,140,0)],   label='Escalera (DINO)'),
        mpatches.Patch(facecolor=[c/255 for c in (148,0,211)],   label='Rampa (DINO)'),
    ]
    ax1.legend(
        handles=patches_cv + patches_dino,
        loc='lower right', fontsize=7, framealpha=0.85, ncol=2,
        title='OpenCV + DINO'
    )

plt.suptitle(f'{NOMBRE_PROYECTO} — Capa 1 Geométrica  |  {pdf_name}',
             fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()

# PNG combinado: {BASENAME}.png
fname_combined = f'{BASENAME}.png'
plt.savefig(fname_combined, dpi=100, bbox_inches='tight')
plt.show()
print(f'✓ PNG combinado guardado: {fname_combined}')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 6 — Informe en consola + guardar JSON
# ══════════════════════════════════════════════════════════
import json
from datetime import datetime

SEP  = '=' * 66
SEP2 = '-' * 66

print(SEP)
print('  ARCHICHECK — INFORME CAPA 1 GEOMETRICA')
print(SEP)
print(f'  Proyecto: {NOMBRE_PROYECTO}')
print(f'  Archivo : {pdf_name}')
print(f'  Fecha   : {datetime.now().strftime("%d/%m/%Y %H:%M")}')
print(f'  Páginas : {" + ".join(str(p["pagina"]) for p in resultados_paginas)}')

for res in resultados_paginas:
    pag      = res['pagina']
    escala   = res['escala']
    tabla    = res['mediciones_geometricas']
    incs     = res['incumplimientos_geo']
    analisis = res['analisis_semantico']
    dino_els = res.get('elementos_dino', [])

    print(f'\n  {SEP2}')
    print(f'  PAGINA {pag} [{res["fname_tag"]}]  |  {escala}  |  {analisis.get("tipo_plano")} — {analisis.get("uso_del_proyecto")}')
    print(f'  Nivel: {analisis.get("nivel")}')
    print(f'  {SEP2}')

    print(f"  {'ID':<7} {'Nombre':<26} {'Tipo':<12} {'Area m2':>8} {'Ancho m':>8}  Estado")
    print(f"  {'--':<7} {'------':<26} {'----':<12} {'-------':>8} {'-------':>8}  ------")
    for f in tabla:
        st  = 'INCUMPLE' if not f['cumple_geo'] else 'OK'
        aw  = str(f['ancho_min_m']) if f['ancho_min_m'] else '-'
        nom = f['nombre'][:25]
        print(f"  {f['id']:<7} {nom:<26} {f['tipo']:<12} {f['area_m2']:>8.2f} {aw:>8}  {st}")
    print(f"  {'':7} {'TOTAL':26} {'':12} {res['total_area_m2']:>8.2f}")

    if incs:
        print(f'\n  INCUMPLIMIENTOS GEOMETRICOS ({len(incs)})')
        print(f'  {SEP2}')
        for inc in incs:
            tipo_inc = inc['tipo'].upper()
            unidad   = 'm2' if inc['tipo'] == 'area' else 'm'
            print(f"  [{tipo_inc}]  {inc['id']} {inc['recinto']}")
            print(f"    Medido: {inc['medido']} {unidad}  |  Minimo: {inc['minimo']} {unidad}  |  Deficit: {inc['deficit']} {unidad}")
            print(f"    Ref: {inc['ref']}")

    inc_sem = analisis.get('incumplimientos_oguc', [])
    if inc_sem:
        print(f'\n  OBSERVACIONES NORMATIVAS CLAUDE ({len(inc_sem)})')
        for inc in inc_sem:
            g = inc.get('gravedad', '?')
            d = inc.get('descripcion', '')[:80]
            print(f"  [{g}] {inc.get('articulo', '')} — {d}")

    if dino_els:
        conteo_dino = {}
        for e in dino_els:
            conteo_dino[e['tipo']] = conteo_dino.get(e['tipo'], 0) + 1
        print(f'\n  ELEMENTOS DINO ({len(dino_els)}): {conteo_dino}')
        angostas = [e for e in dino_els if e['tipo'] == 'puerta' and e['ancho_m'] < 0.90]
        esc_ang  = [e for e in dino_els if e['tipo'] == 'escalera' and e['ancho_m'] < 1.20]
        if angostas:
            print(f"    ⚠ {len(angostas)} puerta(s) < 0.90 m (OGUC Art. 4.2.2)")
        if esc_ang:
            print(f"    ⚠ {len(esc_ang)} escalera(s) < 1.20 m (OGUC Art. 4.2.4)")

print(f'\n{SEP}')

# ── Conteos globales DINO ──────────────────────────────────
puertas_n   = sum(sum(1 for e in r.get('elementos_dino',[]) if e['tipo']=='puerta')    for r in resultados_paginas)
ventanas_n  = sum(sum(1 for e in r.get('elementos_dino',[]) if e['tipo']=='ventana')   for r in resultados_paginas)
escaleras_n = sum(sum(1 for e in r.get('elementos_dino',[]) if e['tipo']=='escalera')  for r in resultados_paginas)
rampas_n    = sum(sum(1 for e in r.get('elementos_dino',[]) if e['tipo']=='rampa')     for r in resultados_paginas)

# Fallback: si DINO no detectó nada, usar conteos del análisis semántico Claude
fuente_conteo = 'dino'
if puertas_n == 0 and ventanas_n == 0 and escaleras_n == 0 and rampas_n == 0:
    for r in resultados_paginas:
        sem = r.get('analisis_semantico', {}).get('elementos_detectados', {})
        puertas_n   += sem.get('puertas', 0)
        ventanas_n  += sem.get('ventanas', 0)
        escaleras_n += sem.get('escaleras', 0)
    if puertas_n + ventanas_n + escaleras_n > 0:
        fuente_conteo = 'semantico'
        print(f'  (conteos desde análisis semántico Claude — DINO no detectó elementos)')
        print(f'  Puertas semántico: {puertas_n}  Ventanas: {ventanas_n}  Escaleras: {escaleras_n}')

resultado_final = {
    'fuente'          : 'colab_opencv_multipagina',
    'proyecto'        : NOMBRE_PROYECTO,
    'archivo'         : pdf_name,
    'fecha'           : datetime.now().isoformat(),
    'basename'        : BASENAME,
    'dpi'             : DPI,
    'paginas'         : resultados_paginas,
    'resumen_global'  : {
        'paginas_analizadas'       : len(resultados_paginas),
        'total_area_m2'            : round(sum(p['total_area_m2'] for p in resultados_paginas), 1),
        'total_recintos'           : sum(len(p['mediciones_geometricas']) for p in resultados_paginas),
        'incumplimientos_geo_total': sum(len(p['incumplimientos_geo']) for p in resultados_paginas),
        'puertas_detectadas'       : puertas_n,
        'ventanas_detectadas'      : ventanas_n,
        'escaleras_detectadas'     : escaleras_n,
        'rampas_detectadas'        : rampas_n,
        'fuente_conteo_elementos'  : fuente_conteo,
    }
}

# JSON: {BASENAME}.json → archicheck_geometrico_pdv_30jun_1729.json
fname_json = f'{BASENAME}.json'
with open(fname_json, 'w', encoding='utf-8') as f:
    json.dump(resultado_final, f, ensure_ascii=False, indent=2)

rg = resultado_final['resumen_global']
print(f'\n✓ JSON guardado: {fname_json}')
print(f'  Paginas   : {rg["paginas_analizadas"]}')
print(f'  Area total: {rg["total_area_m2"]} m2')
print(f'  Recintos  : {rg["total_recintos"]}')
print(f'  Incumpl.  : {rg["incumplimientos_geo_total"]}')
print(f'  Elementos ({rg["fuente_conteo_elementos"]}) — Puertas:{rg["puertas_detectadas"]}  Ventanas:{rg["ventanas_detectadas"]}  Escaleras:{rg["escaleras_detectadas"]}  Rampas:{rg["rampas_detectadas"]}')

In [ ]:
# ══════════════════════════════════════════════════════════
# CELDA 7 — Descargar resultados a tu PC
# ══════════════════════════════════════════════════════════
from google.colab import files

print(f'Proyecto: {NOMBRE_PROYECTO}  |  {BASENAME}')
print()

print(f'Descargando JSON → {BASENAME}.json')
files.download(f'{BASENAME}.json')

print(f'Descargando PNG combinado → {BASENAME}.png')
files.download(f'{BASENAME}.png')

print('Descargando PNGs individuales por planta...')
for vz in viz_pages:
    fname = f'{BASENAME}_{vz["fname_tag"]}.png'
    files.download(fname)
    print(f'  → {fname}')

print()
print('✓ Revisa tu carpeta de Descargas.')
print()
print('Siguiente: en ArchiCheck web sube el PDF + el JSON')
print('+ los PNG de cada planta como archivos adicionales.')